# Package preparation

In [ ]:
!pip install transformers
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


## Import necessary packages

In [ ]:
import os # file management

import torch
from torch import nn, Tensor # neural network
from torch.nn import functional as F

# numerical matrix processing
import numpy as np
from numpy import ndarray

# data/parameter loading
import pandas as pd
import pickle

# visualization
from tqdm.notebook import trange, tqdm

# transfomers
from transformers import AutoTokenizer, AutoModel
from transformers.modeling_outputs import BaseModelOutputWithPoolingAndCrossAttentions

# code instruction
from typing import Union, List, Dict
# filter out warnings
import warnings
warnings.filterwarnings('ignore')

## Some useful functions

In [ ]:
# Utils
def save_parameter(save_object, save_file):
    with open(save_file, 'wb') as f:
        pickle.dump(save_object, f, protocol=pickle.HIGHEST_PROTOCOL)

def load_parameter(load_file):
    with open(load_file, 'rb') as f:
        output = pickle.load(f)
    return output

def batch2device(batch, device):
    """
    Transfer batch of training to GPU/CPU
    Args:
        batch: Dict[str, Tensor], represent for transformer input (input_ids, attention_mask)
        device: torch.device, GPU or CPU
    """
    for key, value in batch.items():
        batch[key] = batch[key].to(device)
    return batch

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# GPU accelerator
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Data preparation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
work_path = "/content/drive/MyDrive/PaperRecommendation/"
checkpoint_path = work_path + "checkpoint/"

In [ ]:
data_train = pd.read_csv(checkpoint_path + "sub_processed_data/train.csv", encoding = "ISO-8859-1")

data_validate = pd.read_csv(checkpoint_path + "sub_processed_data/validate.csv", encoding = "ISO-8859-1")

data_test = pd.read_csv(checkpoint_path + "sub_processed_data/test.csv", encoding = "ISO-8859-1")

data_aims = pd.read_csv(checkpoint_path + "sub_processed_data/aims.csv", encoding = "ISO-8859-1")

data_train.fillna("", inplace=True)
data_validate.fillna("", inplace=True)
data_test.fillna("", inplace=True)
data_aims.fillna("", inplace=True)

n_classes = len(data_aims)

## Feature selection

In [ ]:
X_train = data_train['paper_text'].tolist()
X_valid = data_validate['paper_text'].tolist()
X_test = data_test['paper_text'].tolist()
X_aims = data_aims["scope_text"].tolist()

Y_train = data_train['label'].tolist()
Y_validate = data_validate['label'].tolist()
Y_test = data_test['label'].tolist()

## Tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
train_encodings = tokenizer(
    X_train,
    truncation=True,
    padding="max_length",
    max_length=300,
    return_tensors="pt"
)
valid_encodings = tokenizer(
    X_valid,
    truncation=True,
    padding="max_length",
    max_length=300,
    return_tensors="pt"
)
test_encodings = tokenizer(
    X_test,
    truncation=True,
    padding="max_length",
    max_length=300,
    return_tensors="pt"
)

# save_parameter(train_encodings, checkpoint_path + "pickle/distilroberta_training_encodings.pickle")
# save_parameter(valid_encodings, checkpoint_path + "pickle/distilroberta_valid_encodings.pickle")
# save_parameter(test_encodings, checkpoint_path + "pickle/distilroberta_test_encodings.pickle")

# train_encodings = load_parameter(checkpoint_path + "pickle/distilroberta_training_encodings.pickle")
# valid_encodings = load_parameter(checkpoint_path + "pickle/distilroberta_valid_encodings.pickle")
# test_encodings = load_parameter(checkpoint_path + "pickle/distilroberta_test_encodings.pickle")

## Data loader

In [ ]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        x = {
            key: torch.tensor(val[idx]) for key, val in self.encodings.items()
        }
        y = torch.tensor(self.labels[idx])
        return x, y
    def __len__(self):
        return len(self.labels)

In [ ]:
# Dataset
train_dataset = Dataset(train_encodings, Y_train)
valid_dataset = Dataset(valid_encodings, Y_validate)
test_dataset = Dataset(test_encodings, Y_test)

In [ ]:
# Data loaders
train_loader = torch.utils.data.DataLoader(train_dataset,
                                         batch_size=1,
                                         shuffle=True)
valid_loader = torch.utils.data.DataLoader(valid_dataset,
                                         batch_size=1,
                                         shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset,
                                         batch_size=1,
                                         shuffle=False)

# Model definition

## Sentence Embedder

In [ ]:
class ModelForSE(nn.Module):
    def __init__(self, model_name_or_path):
        super(ModelForSE, self).__init__()
        '''
        Model for sentence embedding
        '''
        self.bert = AutoModel.from_pretrained(model_name_or_path)


    def forward(self,
        input_ids=None,
        attention_mask=None,
        token_type_ids=None,
        position_ids=None,
        head_mask=None,
        inputs_embeds=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
        mlm_input_ids=None,
        mlm_labels=None,
    ):
        outputs = self.bert(
            input_ids,
            attention_mask=attention_mask,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            return_dict=return_dict,
        )

        return BaseModelOutputWithPoolingAndCrossAttentions(
            last_hidden_state=outputs.last_hidden_state,
            hidden_states=outputs.hidden_states,
        )


## Load fine-tuned LM

In [ ]:
# Fine-tuned LM checkpoint (by contrastive learning)

#checkpoint_cl = torch.load(checkpoint_path + "saved_model/Epoch:09 SupCL-DistilRoBERTa.pth") # GPU
checkpoint_cl = torch.load(checkpoint_path + "saved_model/Epoch:09 SupCL-DistilRoBERTa.pth", map_location=torch.device("cpu")) # CPU


# Baseline model of sentence encodings
model_args = {
    "model_name_or_path": "distilroberta-base",
}
base_model = ModelForSE(**model_args)
base_model.load_state_dict(checkpoint_cl["model_state_dict"])
#base_model.bert.gradient_checkpointing_enable()
base_model.to(device)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: distilroberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ModelForSE(
  (bert): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-5): 6 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((76

## Cross-Encoder with Cross-Attention for Journal Recommendation

In [ ]:
class CrossAttentionModel(nn.Module):
    def __init__(self, base_model, hidden_size=768, num_heads=8):
        super().__init__()
        self.base_model = base_model
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=num_heads,
            batch_first=True
        )
        self.scorer = nn.Linear(hidden_size, 1)

    def forward(self, paper_inputs, journal_inputs):
        # Encode paper & journal (token-level)
        paper_out = self.base_model(**paper_inputs)
        journal_out = self.base_model(**journal_inputs)

        Q = paper_out.last_hidden_state      # [bs, Lp, d]
        K = journal_out.last_hidden_state    # [bs, Lj, d]
        V = journal_out.last_hidden_state

        attn_out, _ = self.cross_attn(Q, K, V)

        # Interaction pooling
        pooled = attn_out.mean(dim=1)

        score = self.scorer(pooled)
        return score


In [ ]:
model = CrossAttentionModel(base_model)
model.to(device)

CrossAttentionModel(
  (base_model): ModelForSE(
    (bert): RobertaModel(
      (embeddings): RobertaEmbeddings(
        (word_embeddings): Embedding(50265, 768, padding_idx=1)
        (token_type_embeddings): Embedding(1, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (position_embeddings): Embedding(514, 768, padding_idx=1)
      )
      (encoder): RobertaEncoder(
        (layer): ModuleList(
          (0-5): 6 x RobertaLayer(
            (attention): RobertaAttention(
              (self): RobertaSelfAttention(
                (query): Linear(in_features=768, out_features=768, bias=True)
                (key): Linear(in_features=768, out_features=768, bias=True)
                (value): Linear(in_features=768, out_features=768, bias=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (output): RobertaSelfOutput(
                (dense): Linear(in_featur

In [ ]:
print("Model summary:\n")
print(">> Total params: ", count_parameters(model))

Model summary:

>> Total params:  84481537


# Training

Firstly, we encode Aims&Scopes into the embedding features as external features for training

In [ ]:
aims_encodings = {k: v.to(device) for k, v in tokenizer(
    X_aims,
    truncation=True,
    padding="max_length",
    max_length=300,
    return_tensors="pt"
).items()}


In [ ]:
import torch.nn.functional as F

# ===== Bi-encoder: encode all journal aims (CLS) =====
base_model.eval()
aims_emb = []

with torch.no_grad():
    for j in range(n_classes):
        inputs = {
            "input_ids": aims_encodings["input_ids"][j].unsqueeze(0),
            "attention_mask": aims_encodings["attention_mask"][j].unsqueeze(0),
        }
        out = base_model(**inputs)
        emb = out.last_hidden_state[:, 0, :]   # CLS
        aims_emb.append(emb)

aims_emb = torch.cat(aims_emb, dim=0)          # [n_classes, d]
aims_emb = F.normalize(aims_emb, dim=1)


## Optimizer and Loss function

In [ ]:
# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer=optimizer, gamma=0.96)

# Loss function
loss_fn = nn.CrossEntropyLoss().to(device)

## Training settings

In [ ]:
max_epochs = 7
topks = [1, 3, 5, 10]
history = {
    "train_loss": [],
    "val_loss": [],
    "train_acc@k": [],
    "val_acc@k": [],
}
min_valid_loss = np.inf

## Training loop

In [ ]:
M = 10  # số candidate

In [ ]:
print(model.config.gradient_checkpointing)


AttributeError: 'CrossAttentionModel' object has no attribute 'config'

In [ ]:
# ==============================
# EARLY STOPPING SETUP
# ==============================
best_val_mean_rank = float("inf")
patience = 2              # train nhanh hơn
wait = 0
min_delta = 1.0           # chỉ coi là cải thiện nếu giảm >= 1 rank

for epoch in range(max_epochs):

    train_loss = 0.0
    valid_loss = 0.0

    num_correct_at_k = {
        "train": {k: 0 for k in topks},
        "val": {k: 0 for k in topks}
    }

    # =======================
    # TRAINING
    # =======================
    model.train()
    train_loop = tqdm(train_loader, leave=True)

    for features, labels in train_loop:

        features = batch2device(features, device)
        labels = labels.to(device)

        # ----- Retrieval (TAKS) -----
        with torch.no_grad():
            paper_out = base_model(**features)
            paper_emb = F.normalize(paper_out.last_hidden_state[:, 0, :], dim=1)
            sim_scores = torch.matmul(paper_emb, aims_emb.T)
            candidate_indices = torch.topk(sim_scores, k=M, dim=1).indices[0]

        orig_label = labels.item()

        if orig_label not in candidate_indices:
            continue

        new_label = (candidate_indices == orig_label).nonzero(as_tuple=True)[0].item()
        labels = torch.tensor([new_label], device=device)

        # ----- Cross-Attention Re-ranking -----
        scores = []
        for j in candidate_indices:
            journal_batch = {
                "input_ids": aims_encodings["input_ids"][j].unsqueeze(0).to(device),
                "attention_mask": aims_encodings["attention_mask"][j].unsqueeze(0).to(device),
            }
            score = model(features, journal_batch)
            scores.append(score)

        scores = torch.cat(scores, dim=1)

        optimizer.zero_grad()
        loss = loss_fn(scores, labels)
        loss.backward()
        optimizer.step()

        # ----- Accuracy@K (training) -----
        probs_des = torch.argsort(scores, dim=1, descending=True)

        for k in topks:
            if labels[0] in probs_des[0, :k]:
                num_correct_at_k["train"][k] += 1

        train_loss += loss.item()

        train_loop.set_description(f"Epoch {epoch} - Training")
        train_loop.set_postfix(loss=loss.item())

    train_loss /= len(train_loader)
    history["train_loss"].append(train_loss)
    history["train_acc@k"].append(
        {k: v / len(X_train) for k, v in num_correct_at_k["train"].items()}
    )

    # =======================
    # VALIDATION
    # =======================
    model.eval()
    val_ranks = []
    valid_loop = tqdm(valid_loader, leave=True)

    with torch.no_grad():
        for features, labels in valid_loop:

            features = batch2device(features, device)
            labels = labels.to(device)

            # ----- Retrieval -----
            paper_out = base_model(**features)
            paper_emb = F.normalize(paper_out.last_hidden_state[:, 0, :], dim=1)
            sim_scores = torch.matmul(paper_emb, aims_emb.T)
            candidate_indices = torch.topk(sim_scores, k=M, dim=1).indices[0]

            orig_label = labels.item()

            if orig_label not in candidate_indices:
                continue

            new_label = (candidate_indices == orig_label).nonzero(as_tuple=True)[0].item()
            labels = torch.tensor([new_label], device=device)

            # ----- Cross-Attention -----
            scores = []
            for j in candidate_indices:
                journal_batch = {
                    "input_ids": aims_encodings["input_ids"][j].unsqueeze(0).to(device),
                    "attention_mask": aims_encodings["attention_mask"][j].unsqueeze(0).to(device),
                }
                score = model(features, journal_batch)
                scores.append(score)

            scores = torch.cat(scores, dim=1)

            loss = loss_fn(scores, labels)
            valid_loss += loss.item()

            probs_des = torch.argsort(scores, dim=1, descending=True)

            # ---- Accuracy@K ----
            for k in topks:
                if labels[0] in probs_des[0, :k]:
                    num_correct_at_k["val"][k] += 1

            # ---- Mean Rank ----
            rank_position = (probs_des[0] == labels[0]).nonzero(as_tuple=True)[0].item() + 1
            val_ranks.append(rank_position)

            valid_loop.set_description(f"Epoch {epoch} - Validating")

    valid_loss /= len(valid_loader)

    if len(val_ranks) > 0:
        current_val_mean_rank = sum(val_ranks) / len(val_ranks)
    else:
        current_val_mean_rank = float("inf")

    history["val_loss"].append(valid_loss)
    history["val_acc@k"].append(
        {k: v / len(X_valid) for k, v in num_correct_at_k["val"].items()}
    )

    print(
        f">> Epoch {epoch} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {valid_loss:.4f} | "
        f"Val Mean Rank: {current_val_mean_rank:.2f}"
    )

    # =======================
    # EARLY STOPPING (Mean Rank)
    # =======================
    if best_val_mean_rank - current_val_mean_rank >= min_delta:
        best_val_mean_rank = current_val_mean_rank
        wait = 0

        print(f"Mean Rank improved → {best_val_mean_rank:.2f}. Saving model.")

        saved_path = checkpoint_path + "weight/"
        os.makedirs(saved_path, exist_ok=True)

        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "epoch": epoch,
                "history": history
            },
            saved_path + "Best_MeanRank_DistilRoberta_CrossAttn.pth"
        )
    else:
        break
        """wait += 1
        print(f"Mean Rank did not improve. Wait {wait}/{patience}")

        if wait >= patience:
            print(f"Early stopping triggered at epoch {epoch}.")
            break"""

    lr_scheduler.step()


  0%|          | 0/1000 [00:00<?, ?it/s]

CheckpointError: torch.utils.checkpoint: A different number of tensors was saved during the original forward and recomputation.
Number of tensors saved during forward: 8875
Number of tensors saved during recomputation: 31.

Tip: To see a more detailed error message, either pass `debug=True` to
`torch.utils.checkpoint.checkpoint(...)` or wrap the code block
with `with torch.utils.checkpoint.set_checkpoint_debug_enabled(True):` to
enable checkpoint‑debug mode globally.


# Testing

In [ ]:
print('Số candidate: M = ', M)

In [ ]:
# load model checkpoint for testing
checkpoint = torch.load(checkpoint_path + "weight/Best_Top10_DistilRoberta_CrossAttn.pth")

model = CrossAttentionModel(base_model)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)

history = checkpoint['history']

In [ ]:
# ===== Re-ranking metrics =====
rank_taks_list = []
rank_d_list = []
hit_taks = {1: 0, 3: 0, 5: 0, 10: 0}
hit_d = {1: 0, 3: 0, 5: 0, 10: 0}
num_rerank_samples = 0

test_loop = tqdm(test_loader, leave=True)

with torch.no_grad():
    model.eval()

    for features, labels in test_loop:

        features, labels = batch2device(features, device), labels.to(device)

        # ===== Label gốc =====
        orig_label = labels.item()

        # ===== Stage 1: TAKS retrieval =====
        paper_out = base_model(**features)
        paper_emb = F.normalize(paper_out.last_hidden_state[:, 0, :], dim=1)
        sim_scores = torch.matmul(paper_emb, aims_emb.T)
        candidate_indices = torch.topk(sim_scores, k=M, dim=1).indices[0]

        # ===== Rank của TAKS trong Top-M =====
        pos = (candidate_indices == orig_label).nonzero(as_tuple=True)[0]
        if len(pos) == 0:
            continue   # nếu TAKS không recall được → bỏ sample

        rank_taks = pos.item() + 1
        rank_taks_list.append(rank_taks)

        for k in [1, 3, 5, 10]:
            if rank_taks <= k:
                hit_taks[k] += 1

        # ===== Remap label cho D =====
        new_label = pos.item()
        labels = torch.tensor([new_label], device=device)

        # ===== Stage 2: Cross-Attention Re-ranking =====
        scores = []
        for j in candidate_indices:
            journal_batch = {
                "input_ids": aims_encodings["input_ids"][j].unsqueeze(0).to(device),
                "attention_mask": aims_encodings["attention_mask"][j].unsqueeze(0).to(device),
            }
            score = model(features, journal_batch)
            scores.append(score)

        scores = torch.cat(scores, dim=1)  # [1, M]

        # ===== Rank sau re-ranking =====
        rerank_order = torch.argsort(scores, dim=1, descending=True)[0]
        rank_d = (rerank_order == new_label).nonzero(as_tuple=True)[0].item() + 1
        rank_d_list.append(rank_d)

        for k in [1, 3, 5, 10]:
            if rank_d <= k:
                hit_d[k] += 1

        num_rerank_samples += 1


# Final results

In [ ]:
print("\n===== RE-RANKING RESULTS (TAKS vs TAKS + D) =====")

print(f"Number of evaluated samples: {num_rerank_samples}")

print(f"Mean Rank (TAKS): {sum(rank_taks_list)/len(rank_taks_list):.2f}")
for k in [1, 3, 5, 10]:
    print(f"Hit@{k} TAKS: {hit_taks[k]/num_rerank_samples:.4f}")

print(f"Mean Rank (TAKS + D): {sum(rank_d_list)/len(rank_d_list):.2f}")
for k in [1, 3, 5, 10]:
    print(f"Hit@{k} TAKS + D: {hit_d[k]/num_rerank_samples:.4f}")